# IBRIDO CON NLL

---

In [1]:
import gc
import multiprocessing as mp
import os
import time
from datetime import datetime
from itertools import product

import h5py
import numpy as np
import pandas as pd

from ml_odmr.ODMR_CNN.cnn_model import CNNPredictor
from ml_odmr.FitHybridLikelihood import worker_chunk_hybrid_nll
from ml_odmr.PoissonNoise import PoissonNoise

# =========================================================================
# 1. PARAMETRI DELLA GRIGLIA, IPERPARAMETRI E SETUP PERCORSI
# =========================================================================
list_contrast = [0.8, 0.5, 0.2, 0.1, 0.08, 0.05, 0.02, 0.01]
list_Nph = [3e5, 6e5, 12e5, 24e5, 48e5, 60e5]

mat_file_path = "../data/10k/odmr_easyspin_raw_10000_DOPPIA_DENSITÀ.mat"

# Cartelle separate: riutilizza 'model' se già calcolato, salva il nuovo ibrido in 'hybrid_likelihood'
dir_model_base = os.path.abspath("../benchmarks/10k/model")
dir_hybrid_nll_base = os.path.abspath("../benchmarks/10k/hybrid_likelihood")
os.makedirs(dir_model_base, exist_ok=True)
os.makedirs(dir_hybrid_nll_base, exist_ok=True)

random_seed = 42

# Parametri fit ibrido NLL e parallelizzazione (sicuri per 8 GB RAM)
W0_NORM = 0.025
POS_WINDOW_NORM = 0.05   # Finestra locale ±10 MHz attorno a nu1/nu2 CNN (metti None per bounds [0, 1])
FALLBACK_TO_CNN = True   # Se MIGRAD non converge, tiene la predizione della CNN
N_WORKERS = 4
N_CHUNKS = 16            # 16 blocchi da 625 spettri ciascuno

# Caricamento dati dal file .mat
with h5py.File(mat_file_path, "r") as f:
    spectra_data = np.array(f["spectra_data"], dtype=np.float32).T
    mw_freqs = np.array(f["mw_freqs"]).squeeze()
    labels_c = np.array(f["labels_c"], dtype=np.float64).squeeze()
    labels_s = np.array(f["labels_s"], dtype=np.float64).squeeze()

f_ghz = np.asarray(mw_freqs, dtype=np.float64)
f_mhz = f_ghz * 1000.0
f_min, f_max = float(f_ghz.min()), float(f_ghz.max())
delta_f = f_max - f_min
f_norm_global = np.ascontiguousarray((f_ghz - f_min) / delta_f, dtype=np.float64)

N_spectra_to_run = len(spectra_data)
index_chunks = np.array_split(np.arange(N_spectra_to_run), N_CHUNKS)

noise_gen = PoissonNoise(seed=random_seed)
cnn = None  # Caricato solo se manca almeno un CSV in benchmarks/10k/model

# Dizionari in memoria (come nel notebook ai minimi quadrati)
predictions_model = {"c": {}, "s": {}}
predictions_hybrid_nll = {"c": {}, "s": {}}


# =========================================================================
# 2. FUNZIONI DI SUPPORTO
# =========================================================================
def generate_noisy_batch(contrast_val: float, n_ph_val: float) -> np.ndarray:
    """Genera il batch di spettri rumorosi (conteggi reali) per una coppia (contrast, N_ph)."""
    noisy_batch = np.empty_like(spectra_data, dtype=np.float32)
    for idx in range(N_spectra_to_run):
        ideal_sig = noise_gen.ideal_signal(spectra_data[idx], contrast=contrast_val)
        noisy_batch[idx] = noise_gen.add_noise(
            ideal_sig, n_tot=n_ph_val, seed_modifier=idx
        )
    return noisy_batch


def save_benchmark_csv(filepath: str, df: pd.DataFrame, meta: dict):
    """Salva il DataFrame su disco preceduto dall'header coi metadati."""
    header = "\n".join([f"# {k}: {v}" for k, v in meta.items()]) + "\n"
    with open(filepath, mode="w", encoding="utf-8") as f:
        f.write(header)
        df.to_csv(f, index=False, float_format="%.6f")


# =========================================================================
# 3. ESECUZIONE SULLA GRIGLIA (CNN + FIT IBRIDO NLL)
# =========================================================================
all_combinations = list(product(list_contrast, list_Nph))
total_combos = len(all_combinations)

print(f"Totale combinazioni (Contrast x N_ph): {total_combos}")
print(f"Spettri per combinazione: {N_spectra_to_run} ({len(f_ghz)} punti per spettro)")
print(f"Output Ibrido NLL: '{dir_hybrid_nll_base}'\n")

ctx = mp.get_context("spawn")
t_total_start = time.time()

for combo_idx, (c_val, n_ph) in enumerate(all_combinations, start=1):
    sub_model_dir = os.path.join(dir_model_base, f"contrast={c_val}")
    sub_hyb_dir = os.path.join(dir_hybrid_nll_base, f"contrast={c_val}")
    os.makedirs(sub_model_dir, exist_ok=True)
    os.makedirs(sub_hyb_dir, exist_ok=True)

    file_model = os.path.join(sub_model_dir, f"cnn_Nph{int(n_ph)}.csv")
    file_hybrid_nll = os.path.join(sub_hyb_dir, f"hybrid_nll_Nph{int(n_ph)}.csv")

    if os.path.exists(file_hybrid_nll):
        print(f"[{combo_idx}/{total_combos}] SALTO (già presente): C={c_val}, N_ph={n_ph:.1e}")
        continue

    print(f"[{combo_idx}/{total_combos}] C={c_val:<4} | N_ph={n_ph:.1e} ->", end=" ", flush=True)

    # -----------------------------------------------------------------
    # A. GENERAZIONE RUMORE (stesso identico seed_modifier=idx)
    # -----------------------------------------------------------------
    noisy_sig = generate_noisy_batch(c_val, n_ph)

    # -----------------------------------------------------------------
    # B. PREDIZIONI CNN (caricate da CSV se già salvate, altrimenti inferenza)
    # -----------------------------------------------------------------
    if os.path.exists(file_model):
        df_model = pd.read_csv(file_model, comment="#")
        c_cnn = df_model["c_fit_norm"].to_numpy(dtype=np.float64)
        s_cnn = df_model["s_fit_norm"].to_numpy(dtype=np.float64)
        center_mhz = df_model["c_fit_mhz"].to_numpy(dtype=np.float64)
        delta_mhz = df_model["s_fit_mhz"].to_numpy(dtype=np.float64)
        nu1_mhz = center_mhz - 0.5 * delta_mhz
        nu2_mhz = center_mhz + 0.5 * delta_mhz
        t_cnn_per_spec = float(df_model["fit_time_sec"].mean())
        t_cnn_wall = t_cnn_per_spec * N_spectra_to_run
        mae_c_cnn = float(df_model["err_c_norm"].mean())
        print("CNN: [da cache] |", end=" ", flush=True)
    else:
        if cnn is None:
            cnn = CNNPredictor("ODMR_CNN/artifacts/cnn_50k_48M")

        t_cnn_start = time.perf_counter()
        pred = cnn.predict_counts(noisy_sig, f_mhz)
        t_cnn_wall = time.perf_counter() - t_cnn_start
        t_cnn_per_spec = t_cnn_wall / N_spectra_to_run

        c_cnn = np.asarray(pred["c"], dtype=np.float64)
        s_cnn = np.asarray(pred["s"], dtype=np.float64)
        nu1_mhz = np.asarray(pred["nu1_MHz"], dtype=np.float64)
        nu2_mhz = np.asarray(pred["nu2_MHz"], dtype=np.float64)

        df_model = pd.DataFrame({
            "spectrum_id": np.arange(N_spectra_to_run),
            "converged": True,
            "c_true_norm": labels_c[:N_spectra_to_run],
            "s_true_norm": labels_s[:N_spectra_to_run],
            "c_fit_norm": c_cnn,
            "s_fit_norm": s_cnn,
            "err_c_norm": np.abs(c_cnn - labels_c[:N_spectra_to_run]),
            "err_s_norm": np.abs(s_cnn - labels_s[:N_spectra_to_run]),
            "c_fit_mhz": np.asarray(pred["center_MHz"], dtype=np.float64),
            "s_fit_mhz": np.asarray(pred["delta_MHz"], dtype=np.float64),
            "chi2_red": np.nan,
            "fit_time_sec": t_cnn_per_spec,
        })
        mae_c_cnn = float(df_model["err_c_norm"].mean())

        save_benchmark_csv(
            file_model,
            df_model,
            meta={
                "Benchmark": "1D-CNN Pure Model",
                "Date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "Source_File": os.path.basename(mat_file_path),
                "N_spectra": N_spectra_to_run,
                "N_photons": f"{n_ph:.2e}",
                "Contrast": c_val,
                "Mean_MAE_c_norm": f"{mae_c_cnn:.6f}",
                "Mean_MAE_s_norm": f"{df_model['err_s_norm'].mean():.6f}",
                "Total_Batch_Time_s": f"{t_cnn_wall:.4f}",
            },
        )
        print(f"CNN: {t_cnn_wall:.2f}s |", end=" ", flush=True)

    predictions_model["c"][(c_val, n_ph)] = c_cnn
    predictions_model["s"][(c_val, n_ph)] = s_cnn

    # -----------------------------------------------------------------
    # C. FIT IBRIDO NLL (CNN -> IMINUIT IN PARALLELO A CHUNK)
    # -----------------------------------------------------------------
    t_hyb_start = time.perf_counter()

    tasks = [
        (
            ch,
            noisy_sig[ch],
            nu1_mhz[ch],
            nu2_mhz[ch],
            labels_c[ch],
            labels_s[ch],
            f_norm_global,
            f_min,
            delta_f,
            W0_NORM,
            POS_WINDOW_NORM,
            FALLBACK_TO_CNN,
        )
        for ch in index_chunks
    ]

    # Pool locale chiuso a ogni iterazione: azzera il consumo RAM tra le combinazioni
    with ctx.Pool(processes=N_WORKERS) as pool:
        chunks_out = pool.map(worker_chunk_hybrid_nll, tasks)

    hybrid_results = [item for sublist in chunks_out for item in sublist]
    t_hyb_wall = time.perf_counter() - t_hyb_start

    df_hybrid = pd.DataFrame(hybrid_results)
    # Sommiamo il tempo di inferenza CNN per spettro al tempo di Minuit
    df_hybrid["fit_time_sec"] += t_cnn_per_spec

    predictions_hybrid_nll["c"][(c_val, n_ph)] = df_hybrid["c_fit_norm"].to_numpy()
    predictions_hybrid_nll["s"][(c_val, n_ph)] = df_hybrid["s_fit_norm"].to_numpy()

    n_conv = int(df_hybrid["converged"].sum())
    n_minuit_ok = int(df_hybrid["minuit_valid"].sum())
    n_fallback = int(df_hybrid["used_fallback"].sum())
    succ_rate = (n_conv / N_spectra_to_run) * 100.0
    minuit_rate = (n_minuit_ok / N_spectra_to_run) * 100.0

    save_benchmark_csv(
        file_hybrid_nll,
        df_hybrid,
        meta={
            "Benchmark": "Hybrid (1D-CNN + Poisson NLL iminuit)",
            "Date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "Source_File": os.path.basename(mat_file_path),
            "N_spectra": N_spectra_to_run,
            "N_photons": f"{n_ph:.2e}",
            "Contrast": c_val,
            "Pos_Window_Norm": POS_WINDOW_NORM,
            "Fallback_To_CNN": FALLBACK_TO_CNN,
            "Converged_Spectra": f"{n_conv}/{N_spectra_to_run} ({succ_rate:.2f}%)",
            "Minuit_Valid_Spectra": f"{n_minuit_ok}/{N_spectra_to_run} ({minuit_rate:.2f}%)",
            "Fallback_Count": n_fallback,
            "Mean_MAE_c_norm": f"{df_hybrid['err_c_norm'].mean():.6f}",
            "Mean_MAE_s_norm": f"{df_hybrid['err_s_norm'].mean():.6f}",
            "Total_Wall_Time_s": f"{t_hyb_wall + t_cnn_wall:.2f}",
        },
    )

    print(
        f"Hybrid NLL: {t_hyb_wall:.1f}s (Minuit ok: {minuit_rate:.1f}%) | "
        f"MAE_c(CNN): {mae_c_cnn:.4f} -> MAE_c(Hyb NLL): {df_hybrid['err_c_norm'].mean():.4f}"
    )

    # Pulizia esplicita della RAM prima della combinazione successiva
    del noisy_sig, df_model, tasks, chunks_out, hybrid_results, df_hybrid
    gc.collect()

print(f"\nCompletato in {(time.time() - t_total_start)/60:.1f} minuti!")

Totale combinazioni (Contrast x N_ph): 48
Spettri per combinazione: 10000 (335 punti per spettro)
Output Ibrido NLL: '/Users/zosojack/ml-odmr/benchmarks/10k/hybrid_likelihood'

[1/48] C=0.8  | N_ph=3.0e+05 -> CNN: [da cache] | Hybrid NLL: 4.6s (Minuit ok: 100.0%) | MAE_c(CNN): 0.0002 -> MAE_c(Hyb NLL): 0.0002
[2/48] C=0.8  | N_ph=6.0e+05 -> CNN: [da cache] | Hybrid NLL: 3.5s (Minuit ok: 100.0%) | MAE_c(CNN): 0.0002 -> MAE_c(Hyb NLL): 0.0002
[3/48] C=0.8  | N_ph=1.2e+06 -> CNN: [da cache] | Hybrid NLL: 3.5s (Minuit ok: 100.0%) | MAE_c(CNN): 0.0001 -> MAE_c(Hyb NLL): 0.0001
[4/48] C=0.8  | N_ph=2.4e+06 -> CNN: [da cache] | Hybrid NLL: 3.7s (Minuit ok: 100.0%) | MAE_c(CNN): 0.0001 -> MAE_c(Hyb NLL): 0.0001
[5/48] C=0.8  | N_ph=4.8e+06 -> CNN: [da cache] | Hybrid NLL: 3.9s (Minuit ok: 100.0%) | MAE_c(CNN): 0.0001 -> MAE_c(Hyb NLL): 0.0001
[6/48] C=0.8  | N_ph=6.0e+06 -> CNN: [da cache] | Hybrid NLL: 4.0s (Minuit ok: 100.0%) | MAE_c(CNN): 0.0001 -> MAE_c(Hyb NLL): 0.0001
[7/48] C=0.5  | N_p